In [1]:
pip install flask flask-sqlalchemy

In [ ]:
import os
from flask import Flask, jsonify, request
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__)

# Database Configuration (SQLite default, fallback to Environment Variable)
# In a Colab environment, __file__ is not defined. Use os.getcwd() instead.
BASE_DIR = os.getcwd()
app.config['SQLALCHEMY_DATABASE_URI'] = os.environ.get(
    'DATABASE_URL', f"sqlite:///{os.path.join(BASE_DIR, 'library.db')}"
)
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# -----------------------------------------------------------------------------
# Database Model
# -----------------------------------------------------------------------------
class Book(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(100), nullable=False)
    author = db.Column(db.String(100), nullable=False)
    year = db.Column(db.Integer, nullable=True)

    def to_dict(self):
        """Convert Model Object into JSON-serializable Dictionary."""
        return {
            "id": self.id,
            "title": self.title,
            "author": self.author,
            "year": self.year
        }

# Table initialization
with app.app_context():
    db.create_all()

# -----------------------------------------------------------------------------
# API Endpoints (CRUD)
# -----------------------------------------------------------------------------

# GET: Retrieve all books (with optional search parameter)
@app.route('/books', methods=['GET'])
def get_books():
    search_query = request.args.get('search')
    if search_query:
        books = Book.query.filter(Book.title.ilike(f"%{search_query}%")).all()
    else:
        books = Book.query.all()

    return jsonify([book.to_dict() for book in books]), 200


# GET: Retrieve a single book by ID
@app.route('/books/<int:book_id>', methods=['GET'])
def get_book(book_id):
    book = Book.query.get(book_id)
    if not book:
        return jsonify({"error": "Book not found"}), 404
    return jsonify(book.to_dict()), 200


# POST: Add a new book
@app.route('/books', methods=['POST'])
def add_book():
    data = request.get_json()

    if not data or not data.get('title') or not data.get('author'):
        return jsonify({"error": "Title and Author are required fields"}), 400

    new_book = Book(
        title=data['title'],
        author=data['author'],
        year=data.get('year')
    )

    db.session.add(new_book)
    db.session.commit()

    return jsonify(new_book.to_dict()), 201


# PUT: Update an existing book by ID
@app.route('/books/<int:book_id>', methods=['PUT'])
def update_book(book_id):
    book = Book.query.get(book_id)
    if not book:
        return jsonify({"error": "Book not found"}), 404

    data = request.get_json()
    if not data:
        return jsonify({"error": "Invalid input data"}), 400

    book.title = data.get('title', book.title)
    book.author = data.get('author', book.author)
    book.year = data.get('year', book.year)

    db.session.commit()
    return jsonify(book.to_dict()), 200


# DELETE: Remove a book by ID
@app.route('/books/<int:book_id>', methods=['DELETE'])
def delete_book(book_id):
    book = Book.query.get(book_id)
    if not book:
        return jsonify({"error": "Book not found"}), 404

    db.session.delete(book)
    db.session.commit()
    return jsonify({"message": f"Book ID {book_id} successfully deleted"}), 200

# -----------------------------------------------------------------------------
# App Execution
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # Set debug=False in production environments
    app.run(host='0.0.0.0', port=5000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
